In [1]:
# Import required libraries
import os
import urllib.request
import pandas as pd
import joblib

### Dataset Selection and Loading

In [3]:
os.makedirs("data", exist_ok=True)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

urllib.request.urlretrieve(
    url,
    "data/wdbc.data"
)

print("Dataset downloaded successfully.")

Dataset downloaded successfully.


### Convert wdbc.data to CSV

In [5]:
columns = [
    "id",
    "diagnosis",

    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",

    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",

    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

df = pd.read_csv(
    "data/wdbc.data",
    header=None,
    names=columns
)

df.to_csv(
    "data/breast_cancer.csv",
    index=False
)

print("CSV created successfully.")
print("Shape:", df.shape)

CSV created successfully.
Shape: (569, 32)


### Verify the dataset

In [7]:
print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["diagnosis"].value_counts())
print("\nColumns:")
for i, column in enumerate(df.columns, 1):
    print(i, column)

Dataset shape: (569, 32)
Missing values: 0
Duplicate rows: 0

Class distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Columns:
1 id
2 diagnosis
3 radius_mean
4 texture_mean
5 perimeter_mean
6 area_mean
7 smoothness_mean
8 compactness_mean
9 concavity_mean
10 concave_points_mean
11 symmetry_mean
12 fractal_dimension_mean
13 radius_se
14 texture_se
15 perimeter_se
16 area_se
17 smoothness_se
18 compactness_se
19 concavity_se
20 concave_points_se
21 symmetry_se
22 fractal_dimension_se
23 radius_worst
24 texture_worst
25 perimeter_worst
26 area_worst
27 smoothness_worst
28 compactness_worst
29 concavity_worst
30 concave_points_worst
31 symmetry_worst
32 fractal_dimension_worst


### Prepare features and target

In [9]:
X = df.drop(columns=["id", "diagnosis"])
y = df["diagnosis"]

y = y.map({
    "B": 0,
    "M": 1
})

print("Features:", X.shape)
print("Target:", y.shape)
print("\nTarget classes:")
print(y.value_counts())

Features: (569, 30)
Target: (569,)

Target classes:
diagnosis
0    357
1    212
Name: count, dtype: int64


### Train/Test split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114


### Create test_data.csv

In [13]:
test_data = X_test.copy()

test_data["diagnosis"] = y_test.values

test_data.to_csv(
    "test_data.csv",
    index=False
)

print("test_data.csv created successfully.")
print("Shape:", test_data.shape)

test_data.csv created successfully.
Shape: (114, 31)


### Feature Scaling

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Create and Train the models

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

naive_bayes_model = GaussianNB()

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [18]:
logistic_model.fit(X_train_scaled, y_train)

decision_tree_model.fit(X_train, y_train)

knn_model.fit(X_train_scaled, y_train)

naive_bayes_model.fit(X_train_scaled, y_train)

random_forest_model.fit(X_train, y_train)

print("All models trained successfully.")

All models trained successfully.


### Evaluation Metrics

In [20]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
models = {
    "Logistic Regression": (
        logistic_model,
        X_test_scaled
    ),
    "Decision Tree": (
        decision_tree_model,
        X_test
    ),
    "KNN": (
        knn_model,
        X_test_scaled
    ),
    "Naive Bayes": (
        naive_bayes_model,
        X_test_scaled
    ),
    "Random Forest": (
        random_forest_model,
        X_test
    )
}

In [21]:
results = []

for model_name, (model, test_features) in models.items():

    y_pred = model.predict(test_features)
    y_prob = model.predict_proba(test_features)[:, 1]

    results.append({
        "ML Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

display(results_df)

,ML Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.964912,0.996032,0.975000,0.928571,0.951220,0.924518
1,Decision Tree,0.929825,0.924603,0.904762,0.904762,0.904762,0.849206
2,KNN,0.956140,0.982308,0.974359,0.904762,0.938272,0.905824
3,Naive Bayes,0.921053,0.989087,0.923077,0.857143,0.888889,0.829162
4,Random Forest,0.973684,0.992890,1.000000,0.928571,0.962963,0.944155


### Confusion Matrix

In [23]:
from sklearn.metrics import confusion_matrix

for model_name, (model, test_features) in models.items():

    y_pred = model.predict(test_features)

    cm = confusion_matrix(y_test, y_pred)

    print(model_name)
    print(cm)
    print()

Logistic Regression
[[71  1]
 [ 3 39]]

Decision Tree
[[68  4]
 [ 4 38]]

KNN
[[71  1]
 [ 4 38]]

Naive Bayes
[[69  3]
 [ 6 36]]

Random Forest
[[72  0]
 [ 3 39]]



### Save Models

In [25]:
# import os
# import joblib

os.makedirs("model", exist_ok=True)

joblib.dump(
    logistic_model,
    "model/logistic_regression.pkl"
)

joblib.dump(
    decision_tree_model,
    "model/decision_tree.pkl"
)

joblib.dump(
    knn_model,
    "model/knn.pkl"
)

joblib.dump(
    naive_bayes_model,
    "model/naive_bayes.pkl"
)

joblib.dump(
    random_forest_model,
    "model/random_forest.pkl"
)

joblib.dump(
    scaler,
    "model/scaler.pkl"
)

print("All models saved.")

All models saved.
